# Model Evaluation - Africa Growth Explorer

This notebook evaluates the trained models for predicting GDP per capita growth across African countries.

## 1. Setup and Data Loading

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import json
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
PROJECT_ROOT = Path(r'C:\\dev\\africa-growth-ml')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.features import create_target, select_features_by_coverage, build_feature_matrix, create_temporal_split
from src.train import (
    build_ridge_pipeline, build_hgb_pipeline, train_and_evaluate,
    compute_metrics, global_mean_baseline, persistence_baseline,
    load_pipeline
)
from src.evaluate import (
    compute_metrics_by_group, compute_bootstrap_ci,
    compute_permutation_importance, compute_worst_errors
)
from src.visualization import (
    set_project_style, plot_actual_vs_predicted, plot_residuals,
    plot_feature_importance
)

set_project_style()
np.random.seed(42)

print('Setup complete')

Setup complete


In [2]:
# Load configuration and processed data
config = load_config(PROJECT_ROOT / 'config' / 'indicators.yaml')
panel = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'model_data.parquet')
panel = create_target(panel, config.target_code)

# Feature selection on training data only
feature_cols = [c for c in panel.columns if c not in
               ["iso3", "country_name", "year", "target_next_year"]]
train_mask = panel["year"] <= config.train_end
panel = select_features_by_coverage(panel, feature_cols, min_coverage=0.6,
                                    train_mask=train_mask)

final_features = [c for c in panel.columns if c not in
                 ["iso3", "country_name", "year", "target_next_year"]]

# Split
train, val, test = create_temporal_split(panel, config.train_end, config.val_end)

# Drop test rows with missing current-year growth for fair baseline comparison
if "NY.GDP.PCAP.KD.ZG" in test.columns:
    valid_persistence_mask = test["NY.GDP.PCAP.KD.ZG"].notna()
    n_dropped = (~valid_persistence_mask).sum()
    if n_dropped > 0:
        print(f"Dropping {n_dropped} test rows with missing current-year growth")
        test = test[valid_persistence_mask]

X_train, y_train = build_feature_matrix(train, final_features)
X_val, y_val = build_feature_matrix(val, final_features)
X_test, y_test = build_feature_matrix(test, final_features)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Features ({len(final_features)}): {final_features}")

Dropping 8 test rows with missing current-year growth
Train: (905, 14), Val: (150, 14), Test: (150, 14)
Features (14): ['BX.KLT.DINV.WD.GD.ZS', 'EG.ELC.ACCS.ZS', 'FP.CPI.TOTL.ZG', 'FS.AST.PRVT.GD.ZS', 'IT.NET.USER.ZS', 'NE.CON.GOVT.ZS', 'NE.GDI.TOTL.ZS', 'NE.TRD.GNFS.ZS', 'NY.GDP.PCAP.CD', 'NY.GDP.PCAP.KD.ZG', 'SL.UEM.TOTL.ZS', 'SP.DYN.LE00.IN', 'SP.POP.GROW', 'SP.URB.TOTL.IN.ZS']


## 2. Baseline Comparison

In [3]:
# Global mean baseline
gm_pred = global_mean_baseline(y_train, len(y_test))
gm_metrics = compute_metrics(y_test.values, gm_pred)
print(f"Global Mean Baseline: MAE={gm_metrics['mae']:.4f}, RMSE={gm_metrics['rmse']:.4f}, R2={gm_metrics['r2']:.4f}, DirAcc={gm_metrics['directional_accuracy']:.4f}")

# Persistence baseline
test_with_target = test.loc[y_test.index]
persistence_growth = test_with_target["NY.GDP.PCAP.KD.ZG"].values
valid_persistence = ~np.isnan(persistence_growth)
if valid_persistence.any():
    pers_pred = persistence_baseline(pd.Series(persistence_growth[valid_persistence]))
    pers_metrics = compute_metrics(y_test.values[valid_persistence], pers_pred)
    print(f"Persistence Baseline: MAE={pers_metrics['mae']:.4f}, RMSE={pers_metrics['rmse']:.4f}, R2={pers_metrics['r2']:.4f}, DirAcc={pers_metrics['directional_accuracy']:.4f}")

Global Mean Baseline: MAE=1.8958, RMSE=2.8379, R2=-0.0005, DirAcc=0.8067
Persistence Baseline: MAE=2.2287, RMSE=4.5209, R2=-1.5392, DirAcc=0.7733


## 3. Ridge Regression Results

In [4]:
log_features = [f for f in config.log_transform_candidates if f in final_features]

ridge = build_ridge_pipeline(alpha=config.ridge_alpha,
                             log_transform_features=log_features,
                             all_feature_names=final_features)
ridge_metrics = train_and_evaluate(ridge, X_train, y_train, X_val, y_val)
print(f"Ridge Validation: MAE={ridge_metrics['mae']:.4f}, RMSE={ridge_metrics['rmse']:.4f}, R2={ridge_metrics['r2']:.4f}, DirAcc={ridge_metrics['directional_accuracy']:.4f}")

# Evaluate on test
ridge.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
ridge_test_pred = ridge.predict(X_test)
ridge_test_metrics = compute_metrics(y_test.values, ridge_test_pred)
print(f"Ridge Test: MAE={ridge_test_metrics['mae']:.4f}, RMSE={ridge_test_metrics['rmse']:.4f}, R2={ridge_test_metrics['r2']:.4f}, DirAcc={ridge_test_metrics['directional_accuracy']:.4f}")

Ridge Validation: MAE=3.9833, RMSE=5.9479, R2=-0.1045, DirAcc=0.5667
Ridge Test: MAE=2.4618, RMSE=3.3579, R2=-0.4008, DirAcc=0.6200


## 4. Gradient Boosting Results

In [5]:
hgb = build_hgb_pipeline(max_iter=config.hgb_max_iter,
                         learning_rate=config.hgb_learning_rate,
                         max_depth=config.hgb_max_depth,
                         random_state=config.random_state)
hgb_metrics = train_and_evaluate(hgb, X_train, y_train, X_val, y_val)
print(f"HGB Validation: MAE={hgb_metrics['mae']:.4f}, RMSE={hgb_metrics['rmse']:.4f}, R2={hgb_metrics['r2']:.4f}, DirAcc={hgb_metrics['directional_accuracy']:.4f}")

# Evaluate on test
hgb.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
hgb_test_pred = hgb.predict(X_test)
hgb_test_metrics = compute_metrics(y_test.values, hgb_test_pred)
print(f"HGB Test: MAE={hgb_test_metrics['mae']:.4f}, RMSE={hgb_test_metrics['rmse']:.4f}, R2={hgb_test_metrics['r2']:.4f}, DirAcc={hgb_test_metrics['directional_accuracy']:.4f}")

HGB Validation: MAE=3.9061, RMSE=5.9417, R2=-0.1022, DirAcc=0.5800


HGB Test: MAE=3.5397, RMSE=4.9984, R2=-2.1039, DirAcc=0.5267


## 5. Model Comparison Table

In [6]:
comparison = pd.DataFrame({
    "Model": ["Global Mean", "Persistence", "Ridge (Val)", "HGB (Val)", "Ridge (Test)", "HGB (Test)"],
    "MAE": [gm_metrics["mae"], pers_metrics["mae"], ridge_metrics["mae"], hgb_metrics["mae"], ridge_test_metrics["mae"], hgb_test_metrics["mae"]],
    "RMSE": [gm_metrics["rmse"], pers_metrics["rmse"], ridge_metrics["rmse"], hgb_metrics["rmse"], ridge_test_metrics["rmse"], hgb_test_metrics["rmse"]],
    "R2": [gm_metrics["r2"], pers_metrics["r2"], ridge_metrics["r2"], hgb_metrics["r2"], ridge_test_metrics["r2"], hgb_test_metrics["r2"]],
    "Directional_Accuracy": [gm_metrics["directional_accuracy"], pers_metrics["directional_accuracy"], ridge_metrics["directional_accuracy"], hgb_metrics["directional_accuracy"], ridge_test_metrics["directional_accuracy"], hgb_test_metrics["directional_accuracy"]]
})
comparison

,Model,MAE,RMSE,R2,Directional_Accuracy
0,Global Mean,1.895766,2.837858,-0.000522,0.806667
1,Persistence,2.228654,4.520915,-1.539206,0.773333
2,Ridge (Val),3.983306,5.947938,-0.104486,0.566667
3,HGB (Val),3.906063,5.941681,-0.102163,0.580000
4,Ridge (Test),2.461787,3.357895,-0.400811,0.620000
5,HGB (Test),3.539652,4.998371,-2.103861,0.526667


## 6. Actual vs. Predicted (Best Model)

In [7]:
# Determine winner
if hgb_test_metrics["mae"] <= ridge_test_metrics["mae"]:
    winner_name = "HistGradientBoostingRegressor"
    winner_pred = hgb_test_pred
    winner_metrics = hgb_test_metrics
else:
    winner_name = "Ridge"
    winner_pred = ridge_test_pred
    winner_metrics = ridge_test_metrics

print(f"Winner: {winner_name}")
print(f"Test MAE: {winner_metrics['mae']:.4f}")

fig, ax = plt.subplots(figsize=(8, 8))
plot_actual_vs_predicted(y_test.values, winner_pred, ax=ax, title=f"Actual vs Predicted - {winner_name} (Test)")
plt.show()
plt.close(fig)

Winner: Ridge
Test MAE: 2.4618


C:\Users\ingex\AppData\Local\Temp\ipykernel_34472\491455572.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Residual Analysis

In [8]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_residuals(y_test.values, winner_pred, ax=ax, title=f"Residuals - {winner_name} (Test)")
plt.show()
plt.close(fig)

C:\Users\ingex\AppData\Local\Temp\ipykernel_34472\2378835807.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Feature Importance

In [9]:
# Load precomputed permutation importance from metadata
with open(PROJECT_ROOT / 'models' / 'model_metadata.json', 'r') as f:
    metadata = json.load(f)

importance = pd.Series(metadata["metrics"]["feature_importance"]).sort_values(ascending=False)
print(importance)

fig, ax = plt.subplots(figsize=(10, 6))
plot_feature_importance(importance, ax=ax, title="Permutation Feature Importance (Test)")
plt.show()
plt.close(fig)

EG.ELC.ACCS.ZS          0.603648
NY.GDP.PCAP.CD          0.221633
SL.UEM.TOTL.ZS          0.176612
FS.AST.PRVT.GD.ZS       0.056488
NE.CON.GOVT.ZS         -0.055565
BX.KLT.DINV.WD.GD.ZS   -0.077802
NE.TRD.GNFS.ZS         -0.079605
NY.GDP.PCAP.KD.ZG      -0.088468
SP.URB.TOTL.IN.ZS      -0.093499
IT.NET.USER.ZS         -0.140919
SP.DYN.LE00.IN         -0.150485
SP.POP.GROW            -0.153790
FP.CPI.TOTL.ZG         -0.195515
NE.GDI.TOTL.ZS         -0.211506
dtype: float64


C:\Users\ingex\AppData\Local\Temp\ipykernel_34472\4213679135.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Bootstrap Confidence Intervals

In [10]:
# Bootstrap CI for MAE
mae_lower, mae_upper = compute_bootstrap_ci(
    y_test.values, winner_pred,
    metric_fn=lambda a, p: np.mean(np.abs(a - p)),
    n_bootstrap=1000
)
print(f"MAE 95% CI: [{mae_lower:.4f}, {mae_upper:.4f}]")

# Bootstrap CI for RMSE
rmse_lower, rmse_upper = compute_bootstrap_ci(
    y_test.values, winner_pred,
    metric_fn=lambda a, p: np.sqrt(np.mean((a - p)**2)),
    n_bootstrap=1000
)
print(f"RMSE 95% CI: [{rmse_lower:.4f}, {rmse_upper:.4f}]")

# Bootstrap CI for Directional Accuracy
dir_lower, dir_upper = compute_bootstrap_ci(
    y_test.values, winner_pred,
    metric_fn=lambda a, p: np.mean((a >= 0) == (p >= 0)),
    n_bootstrap=1000
)
print(f"Directional Accuracy 95% CI: [{dir_lower:.4f}, {dir_upper:.4f}]")

MAE 95% CI: [2.1086, 2.8430]
RMSE 95% CI: [2.7796, 3.9231]
Directional Accuracy 95% CI: [0.5400, 0.6935]


## 10. Error Analysis

In [11]:
# Worst errors
test_preds = pd.DataFrame({
    "iso3": test.loc[y_test.index, "iso3"].values,
    "year": test.loc[y_test.index, "year"].values,
    "country_name": test.loc[y_test.index, "country_name"].values,
    "actual": y_test.values,
    "predicted": winner_pred
})
worst = compute_worst_errors(test_preds, top_n=10)
print("Top 10 Worst Errors:")
print(worst[["iso3", "year", "country_name", "actual", "predicted", "abs_error"]])

# Metrics by country
country_metrics = compute_metrics_by_group(test_preds, group_col="iso3")
country_metrics = country_metrics.sort_values("mae")
print("\nMAE by Country (sorted):")
print(country_metrics[["iso3", "mae", "rmse", "directional_accuracy"]].to_string())

# Metrics by year
year_metrics = compute_metrics_by_group(test_preds, group_col="year")
print("\nMAE by Year:")
print(year_metrics[["year", "mae", "rmse", "directional_accuracy"]].to_string())

Top 10 Worst Errors:
    iso3  year       country_name     actual  predicted  abs_error
33   CPV  2021         Cabo Verde  15.154508  -0.268803  15.423311
73   LBY  2022              Libya   8.965347  -3.440690  12.406037
38   DJI  2023           Djibouti   5.536225  -3.068404   8.604629
37   DJI  2022           Djibouti   5.339196  -2.902315   8.241510
123  SYC  2021         Seychelles  -9.019861  -0.896334   8.123528
64   GNQ  2022  Equatorial Guinea  -9.629959  -1.788047   7.841912
35   CPV  2023         Cabo Verde   6.469916  -0.858032   7.327948
99   NER  2021              Niger   8.631126   1.864432   6.766694
124  SYC  2022         Seychelles   5.272017  -1.243508   6.515525
14   BWA  2023           Botswana  -4.356882   2.016795   6.373677

MAE by Country (sorted):
   iso3       mae       rmse  directional_accuracy
27  MDG  0.127524   0.132989              1.000000
47  ZAF  0.157372   0.174967              1.000000
7   CMR  0.298887   0.374377              1.000000
9   COG  0.3


MAE by Year:
   year       mae      rmse  directional_accuracy
0  2021  2.736863  3.777257                  0.60
1  2022  2.452835  3.389157                  0.62
2  2023  2.195663  2.841184                  0.64


## 11. COVID-19 Placement Note

**Temporal Split Design Decision:**

- **Training**: 2000-2017 (pre-COVID)
- **Validation**: 2018-2020 (includes 2020 COVID shock)
- **Test**: 2021+ (post-COVID)

The 2020 COVID-19 shock falls in the validation period. This means the model selection process is informed by how models handle a major shock year. The test set is entirely post-COVID, which tests generalization after the shock. This is a deliberate decision - we want to select models that are robust to structural breaks, and then evaluate them on the post-shock period.